# Model Forecasting Tutorial

The purpose of this notebook is to demonstrate reading and handling of spatial data and deploying a trained model on a spatial grid to generate predictions.

## Setup

In [ ]:
import xarray as xr
import rioxarray as rxr
from pyproj import Transformer
import tensorflow as tf
import numpy as np
import glob
import re
import os.path as osp
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
from utils import retrieve_url, str2time, read_pkl
from data_funcs import int2fstep
from moisture_rnn_xarray import bands_to_names, names_to_bands, get_file_list, preprocess, calc_eqs, calc_rain, bbox_to_xy

In [ ]:
import importlib
import moisture_rnn_xarray
importlib.reload(moisture_rnn_xarray)
from moisture_rnn_xarray import bands_to_names, names_to_bands, get_file_list, preprocess, calc_eqs, calc_rain

## User Options

In [ ]:
start_time = 2024042000            # start time for predictions (YYYMMDDHH)
end_time = 2024042002              # end time for predictions (YYYMMDDHH)
forecast_step = 3                  # forecast step for HRRR model
base_url = "https://demo.openwfm.org/web/data/fmda/tif/" # base URL for staged data
bbox = [37, -111, 46, -95]         # Spatial bounding box for preds (optional)
data_path = "data"

## Read in Model & Train Data

The fitted model and trained data determine which features should be read for predictions and how data scaling should be performed.

NOTE: run notebook `fmda_rnn_train_and_save.ipynb` to generate these files.

In [ ]:
# Training Data Object
rnn_dat = read_pkl("outputs/models/rnn_data_rocky.pkl")
# Fitted Model
mod = tf.keras.models.load_model("outputs/models/model_predict_rocky.keras")

In [ ]:
mod.summary()

## Determine HRRR Data and Retrieve/Read

Based on user inputs and model specifications, generate a list of files needed to read in to create predictions. 

NOTE: this assumes a standard file structure and naming convention.

In [ ]:
print(f"Features used to train model: {rnn_dat.features_list}")

In [ ]:
bands = names_to_bands(rnn_dat.features_list)
bnames = bands_to_names(bands)

# Manually fix rain variables
if 'rain' in bnames:
    print("Fixing rain variables")
    bands.remove(628)
    bnames.remove("rain")
    bands.append(629)
    bnames.append("precip_accum")

In [ ]:
print(f"Required HRRR Band Numbers: {bands}")
print(f"Required HRRR Band Names: {bnames}")

### Retrieve Data Remotely

Based on the desired bands and forecast step, we create a list of files that are structured in such a way where `xarray` can easily read them and join by the proper dimensions. The dimensions of the resulting data need to include time and feature type, which we will refer to as "band" for now.

In order to calculate rain, we must read in the previous forecast step given by user input so we can calculate the hourly rainfall from the accumulated precipitation.

A small set of test data is staged on Demo, and we will retrieve it unless it already exists.

In [ ]:
fstep = int2fstep(forecast_step)
if forecast_step > 0:
    fprev = int2fstep(forecast_step-1)
print(f"{fstep=}")
print(f"{fprev=}")

In [ ]:
file_list = get_file_list(start_time, end_time, fstep=fstep, bands_list = bands)
file_list

In [ ]:
# Rain file list, used to calculate hourly rainfall
file_list_prev = get_file_list(start_time, end_time, fstep=fprev, bands_list = [629])
file_list_prev

In [ ]:
for sublist in file_list:
    for file in sublist:
        retrieve_url(
            f"{base_url}/{file}",
            dest_path = osp.join(data_path, file)
        )

In [ ]:
for sublist in file_list_prev:
    for file in sublist:
        print(file)
        retrieve_url(
            f"{base_url}/{file}",
            dest_path = osp.join(data_path, file)
        )

## Read Data

Using `xarray`, we read in the files and concatenate by time and feature type. This requires using a preprocessing function, which we named `preprocess` in the custom module. 

We read the data in and do some basic exploration...

In [ ]:
files = [[osp.join(data_path, filename) for filename in sublist] for sublist in file_list]
files_prev = [[osp.join(data_path, filename) for filename in sublist] for sublist in file_list_prev]

data = xr.open_mfdataset(
    files,
    concat_dim=["time", "band"],
    combine="nested",
    preprocess=preprocess
)

data.attrs["forecast_step"] = fstep

data_prev = xr.open_mfdataset(
    files_prev,
    concat_dim=["time", "band"],
    combine="nested",
    preprocess=preprocess
)

data_prev.attrs["forecast_step"] = fprev

In [ ]:
data

In [ ]:
print(data.dims)

In [ ]:
print(data.band_data.shape)

In [ ]:
plt.imshow(data.sel(band = "temp").isel(time=0).band_data)
plt.title(f"HRRR Temperature at {data.time[0].values.astype('M8[ms]').astype(datetime)}")
# plt.colorbar(label="Temperature (°C)")
plt.show()

## Calculate Features

If we build a model based on drying or wetting equilibrium, we must calculate those features from RH and temperature. Additionally, hourly rainfall (mm/hr) must be calculated from accumulated rainfall over the relevant timeperiod. We use custom functions from the module `moisture_rnn_xarray`

In [ ]:
import importlib
import moisture_rnn_xarray
importlib.reload(moisture_rnn_xarray)
from moisture_rnn_xarray import bands_to_names, names_to_bands, get_file_list, preprocess, calc_eqs, calc_rain

In [ ]:
data.band

In [ ]:
data = calc_eqs(data)
data = calc_rain(data, data_prev)

In [ ]:
plt.imshow(data.sel(band = "Ed").isel(time=0).band_data)
plt.title(f"HRRR Drying Eq at {data.time[0].values.astype('M8[ms]').astype(datetime)}")
# plt.colorbar(label="Temperature (°C)")
plt.show()

## Subset to Bounding Box

Given spatial bounding box, create a subsetted dataset. To perform this operation, we extract projection info from just one of the files, assuming that the spatial projection info will be the same across all HRRR files.

TODO: store original index values to know where subset sits in CONUS. Subsetting resets indices

In [ ]:
ds = rxr.open_rasterio(osp.join(data_path, file_list[0][0]))

In [ ]:
# Print Projection Info
ds.rio.transform()

In [ ]:
# Convert bounding box to xy on HRRR data grid
minx, miny, maxx, maxy = bbox_to_xy(bbox, ds.rio.crs)
# Clip spatial xarray
ds_clipped = ds.rio.clip_box(minx=minx, miny=miny, maxx=maxx, maxy=maxy)
# Apply clipped spatial xarray to larger dataset
data2 = data.sel(x=slice(minx, maxx), y=slice(maxy, miny))  # Note: flip y for descending order

In [ ]:
plt.imshow(data2.sel(band = "Ed").isel(time=0).band_data)
plt.title(f"HRRR Drying Eq at {data.time[0].values.astype('M8[ms]').astype(datetime)} - Subset to Rocky GACC")
# plt.colorbar(label="Temperature (°C)")
plt.show()

## Reshape Spatial Data and Apply Model

Steps:
* Extract features based on training data
* Reshape and apply data fitted scaler from training data
* 

In [ ]:
rnn_dat.features_list

In [ ]:
data.band

In [ ]:
Xnew = data.band_data.sel(band=rnn_dat.features_list)
print(Xnew.dims)
print(Xnew.shape)
Xnew = Xnew.stack(spacetime=('x', 'y', 'time'))
Xnew = Xnew.transpose('spacetime', 'band')
print(f"Reshaped Data Shape: {Xnew.shape}")

In [ ]:
np.all(Xnew.band == rnn_dat.features_list).values

In [ ]:
print(f"{rnn_dat.scaler.n_features_in_ = }")

In [ ]:
# Apply scaling (resulting in a numpy array)
Xnew_scaled = rnn_dat.scaler.transform(Xnew)

In [ ]:
# Retrieve original dimensions from Xnew
x, y, time, band = data.sizes['x'], data.sizes['y'], data.sizes['time'], len(rnn_dat.features_list)

# Reshape the scaled array back to the shape (x, y, time, band)
X = Xnew_scaled.reshape(x*y, time, band)

print(X.shape)

# Predict
preds = mod.predict(X)

In [ ]:
# Reshape to Grid and Plot
preds_reshaped = preds.reshape(x, y, time, 1)
preds_da = xr.DataArray(
    preds_reshaped,
    dims=('x', 'y', 'time', 'band'),
    coords={'x': data.coords['x'], 'y': data.coords['y'], 'time': data.coords['time'], 'band': ["preds"]}
)

# # Concatenate preds_da with Xnew along the 'band' dimension
data_preds = xr.concat([data.band_data.sel(band=rnn_dat.features_list), preds_da], dim='band')

In [ ]:
t = 0
plt.figure(figsize=(12, 8))
plt.imshow(data_preds.sel(band="preds").isel(time=t))
plt.title(f"FMC Prediction at time {data.time[t].values.astype('M8[ms]').astype(datetime)}")
plt.colorbar(label="FMC (%)")

In [ ]:
t = 1
plt.figure(figsize=(12, 8))
plt.imshow(data_preds.sel(band="preds").isel(time=t))
plt.title(f"FMC Prediction at time {data.time[t].values.astype('M8[ms]').astype(datetime)}")
plt.colorbar(label="FMC (%)")

In [ ]:
t = 2
plt.figure(figsize=(12, 8))
plt.imshow(data_preds.sel(band="preds").isel(time=t))
plt.title(f"FMC Prediction at time {data.time[t].values.astype('M8[ms]').astype(datetime)}")
plt.colorbar(label="FMC (%)")

In [ ]:
vals = data_preds.sel(band="preds").values

In [ ]:
vals.shape

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(vals[0], vmin=vals.min(), vmax=vals.max())
cbar = fig.colorbar(im, ax=ax, label="FMC (%)")
title = ax.set_title("")

# Update function for animation
def update(t):
    im.set_array(vals[t])  # Update with the next frame
    time_value = data.time[t].values.astype("M8[ms]").astype(datetime)
    title.set_text(f"FMC Prediction at time {time_value}")

# Create the animation
ani = animation.FuncAnimation(fig, update, frames=vals.shape[0], interval=200)

# Save the animation
ani.save("outputs/animation_map.gif", writer=PillowWriter(fps=2))

In [ ]:
from IPython.display import Image
# Image(filename="outputs/animation_map.gif")